## Install Libraries

In [13]:
!pip install phidata openai duckduckgo-search lancedb tantivy pypdf
!pip install pylance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 MB 12.3 MB/s eta 0:00:00


## Loading Libraries and API keys

In [14]:
import os
from google.colab import userdata
OpenAI_API = userdata.get('OpenAI')
os.environ['OPENAI_API_KEY'] = OpenAI_API

In [15]:
from phi.knowledge.pdf import PDFKnowledgeBase, PDFReader
from phi.vectordb.lancedb import LanceDb
from phi.vectordb.search import SearchType
from phi.agent import Agent
from phi.storage.agent.sqlite import SqlAgentStorage
from phi.model.openai import OpenAIChat
from phi.tools.duckduckgo import DuckDuckGo

# VectorDB setup

In [16]:
# Initialize a LanceDb object for vector database operations
vector_db = LanceDb(
    table_name="documents",  # Name of the table to store vector data
    uri="/tmp/lancedb",    # File path for the database storage
    search_type=SearchType.keyword,  # Specify the type of search (e.g., keyword-based search)
)

In [17]:
# Create a directory named 'Data' in the current working directory
! mkdir Data

# Download the GPT-4 paper PDF from the given URL
# Save the downloaded file into the 'Data' directory with the name 'gpt-4.pdf'
! wget "https://cdn.openai.com/papers/gpt-4.pdf" -O Data/gpt-4.pdf

mkdir: cannot create directory ‘Data’: File exists
--2026-05-15 00:31:48--  https://cdn.openai.com/papers/gpt-4.pdf
Resolving cdn.openai.com (cdn.openai.com)... 150.171.109.145, 2603:1061:14:90::1
Connecting to cdn.openai.com (cdn.openai.com)|150.171.109.145|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5229908 (5.0M) [application/pdf]
Saving to: ‘Data/gpt-4.pdf’

Data/gpt-4.pdf      100%[===================>]   4.99M  28.5MB/s    in 0.2s    

2026-05-15 00:31:48 (28.5 MB/s) - ‘Data/gpt-4.pdf’ saved [5229908/5229908]



In [18]:
# Initialize a PDFKnowledgeBase object for managing knowledge extracted from PDF files
pdf_knowledge_base = PDFKnowledgeBase(
    path="/content/Data/",               # Path to the directory containing PDF files
    vector_db=vector_db,                 # Vector database instance for storing and querying extracted information
    reader=PDFReader(chunk=True),        # PDFReader instance with chunking enabled to process documents in smaller parts
)

In [19]:
pdf_knowledge_base.load() # load the data into vector db

INFO     Creating collection

INFO     Loading knowledge base

INFO     Reading: gpt-4

INFO     Added 0 documents to knowledge base

## Agent Setup

In [20]:
agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    knowledge=pdf_knowledge_base,
    tools=[DuckDuckGo()],
    show_tool_calls=True,
    markdown=True,
    storage=SqlAgentStorage(table_name="data", db_file="data.db"),
    add_history_to_messages=True,
)

In [21]:
agent.print_response(
  "Does GPT-4 accept visual inputs?",
  stream=True
)

Output()

In [22]:
agent.print_response(
  "What is the comparison of GPT-4 and GPT-3.5?",
  stream=True
)

Output()